






<table style="background: white;">
    <tr style="background: white;">
        <td> <img src='Figures/hello.png'  width="400px" />   
            <img src='Figures/mpqp.png'  width="400px" />  
        <img src='Figures/colibriTD.png'  width="250px" />
    </tr>
</table>

Dans ce notebook Jupyter, nous allons programmer deux algorithmes quantiques historiques et essayer de comprendre, en codant, ce qui se passe lorsqu’on programme en quantique.

Pour cela, nous allons utiliser la bibliothèque MPQP, développée par la start-up française ColibriTD.

MPQP est une bibliothèque open source en Python. Elle permet de créer des circuits et de les exécuter sur des simulateurs ou sur des machines quantiques (IBM, Azure, AWS, IonQ, etc.)

Les docs et infos sur MPQP sont disponibles sur : https://mpqpdoc.colibri-quantum.com/

## Get ready
On commence par importer les bibilothèques nécessaires à MPQP, les outils de visualisation ainsi que Numpy
```Python
from mpqp import* 
from mpqp.gates import*
from mpqp.measures import BasisMeasure
import numpy as np
```
Nous aurons aussi besoin de la bibliothèque dédiée aux exécutions

```Python
from mpqp.execution import*
```
   

## Petit tour de chauffe 

Commençons par créer un circuit à un qubit
```Python
circuit=QCircuit(1)
```
et plaçons sur ce circuit une porte d'Hadamard,
$$H=\dfrac{1}{\sqrt{2}}\begin{pmatrix} 
1 & 1 \\
1 & -1
\end{pmatrix}$$

```Python
circuit.add(H(0))
```
Enfin afin d'obtenir un résultat, nous devons mesurer l'état quantique obtenu. Pour cela on ajoute donc une mesure

```Python
circuit.add(BasisMeasure())
```

Pour exécuter le circuit, il faut choisir un backend. Ici nous allons travailler avec différents simulateurs (l'utilisation au vrai backend nécessite des crédits)

```Python


from mpqp.execution.simulated_devices import*
result=run(circuit,[IBMDevice.AER_SIMULATOR,GOOGLEDevice.CIRQ_LOCAL_SIMULATOR,IBMSimulatedDevice.FakeBrisbane])
```

Pour représenter les résultats, on peut afficher l'histogramme
```Python
import matplotlib.pyplot
result.plot()
```

Passons maintenant à un circuit à deux qubits:

<table style="background: white;">
    <tr style="background: white;">
        <td> <img src='../Figures/epr.png'  width="400px" />   
    </tr>
</table>

Créons le circuit qui génère cet état et mesurons-le

# Un premier algorithme: L'algorithme de Deutsch (1985)

Le problème : on suppose avoir une urne qui contient deux boules. Les couleurs possibles pour les boules sont rouges ou bleues. Déterminer si les boules ont la même couleur

Il s'agit d'un problème de type "boite noire" ('black-box' ou 'oracle')

<table style="background: white;">
    <tr style="background: white;">
        <td> <img src='Figures/deutsh-urne2.png'  width="400px" />   
    </tr>
</table>

Mathématiquement, nous allons modéliser ceci par une fonction $$f:\{0,1\} \to \{0,1\}$$ et le but est de savoir si la fonction renvoie toujours la même valeur.

Pour cela on définit un oracle qui correspond à une des 4 possibilités:

```Python
def OracleClassique(x,c):
    if c==0:
        return 0
    if c==1:
        return 1
    if c==2:
        return x
    if c==3:
        return (1+x)% 2
```

En choisissant $c$ aléatoirement, on se retrouve dans les conditions de l'expérience

```Python
from random import randint
c=randint(0,3)
```

Maintenant ```OracleClassique(,c)``` est vraiement un oracle et on ne sait pas puisque $c$ a été choisi aléatoirement quelle est la fonction modélisée. En particulier on ne sais pas si la fonction est constante ($c=0$ ou $c=1$) ou si au contraire elle n'est pas constante ($c=2$ ou $c=3$)

## Comment résoudre le problème ?

Classiquement, on peut résoudre le problème de savoir si la fonction est constante ($c=0,1$) ou pas ($c=2,3$) en appelant deux fois l'oracle. Si la différnence des valeurs est nulle, la fonction est constante. Sinon elle ne l'est pas.

## Algorithme Quantique de Deutsch

Pour la version quantique, nous devons définir l'analogue de la fonction Oracle. En quantique, comme les circuits sont des matrices inversibles, il faut, pour modéliser une fonction, la rendre inversible. Donc pour $x\in\{0,1\}$ on cherche une porte quantique qui vérifie:
<table style="background: white;">
    <tr style="background: white;">
        <td> <img src='Figures/circuit.png'  width="800px" />   
    </tr>
</table>

Pour cela complétons le code suivant:
```Python
def OracleDeutsch(c):
    U=QCircuit(2)
    if c==0:
        U.add(...)
    if c==1:
        U.add(...)
    if c==2:
        U.add(...)
    if c==3:
        U.add(...)
    return U
```

## Deutsch Naif
Maintenant que nous avons un oracle essayons de mettre en superposition les valeurs d'entrée avec une porte d'Hadamard:

<table style="background: white;">
    <tr style="background: white;">
        <td> <img src='Figures/deutshnaif.png'  width="800px" />   
    </tr>
</table>

Implémentons cette première version de l'algorithme. Après la porte $U_f$, l'état du système est 
$$ \dfrac{1}{\sqrt{2}}(\ket{0}\ket{f(0)}+\ket{1}\ket{f(1)})$$

Précisons le nombre de shots égal à $1$  dans 
```Python 
BasisMeasure(shots=1))
```
Qu'observe-t-on ?

On ne peut pas conclure... la mesure révèle ici une seule valeur de $f$

## Algorithme Quantique de Deutsch
En 1985 David Deutsch a proposé un algorithme qui résout ce problème plus rapidement,

<table style="background: white;">
    <tr style="background: white;">
        <td> <img src='Figures/deutsch.png'  width="800px" />   
    </tr>
</table>

Testons l'algorithme de Deutsch pour les différents oracles et vérifions que l'algorithme résout le problème initial en une seule évaluation de la fonction

On voit clairement que l'algorithme de Deutsch ne se comporte pas de la même façon pour le cas constant, $c=0$ ou $c=1$, ou le cas non constant, $c=2$ ou $c=3$. Ainsi si on mesure $|0\rangle$ on sait que la fonction de l'oracle était une fonction constante et si on mesure $|1\rangle$, la fonction n'était pas constante. Un seul appel à l'Oracle a été nécessaire pour cela.

### Commentaires

L'algorithme de Deutsch a été publié en 1985 dans un article intitulé "Quantum theory, the Church-Turing principle and the universal quantum computer" (Proceedings of the Royal Society A). Il s'agit du premier algorithme quantique connu. L'algorithme de Deutsch utilise une propriété profondément quantique : l'interférence sur le premier qubit. Cette interférence est distincte si $f$ est constante ou pas et c'est ce motif d'interférence qui est mesuré sur le premier qubit. Très souvent dans les algorithmes quantiques, la superposition des états fait émerger une propriété qui indirectement permet la résolution du problème.

# Algorithme de Bernstein-Vazirani (1997)

Considérons un code secret à $n$ chiffres binaires, par exemple pour $n=6$, prenons $c=c_1c_2c_3c_4c_5c_6$ et supposons qu'on dispose d'une fonction classique $$f_c(x_1,x_2,x_3,x_4,x_5,x_6)=c_1x_1\oplus  c_2x_2\oplus c_3x_3\oplus c_4x_4\oplus c_5x_5\oplus c_6x_6.$$

Une telle fonction est peut-être codée de la manière suivante:

```Python
n=6
c=[randint(0,1) for _ in range(n)]
def f(x,c):
    s=0
    n=len(c)
    for i in range(n):
        if c[i]==1:
            s=s+x[i]
    return(s%2)
```
Comment peut-on trouver $c$ ?

### L'oracle quantique
L'analogue quantique de la fonction $f(x,c)$ peut être défini à partir de CNOT. Par exemple le circuit suivant correspond à $c=[1,0,1,1]$

<table style="background: white;">
    <tr style="background: white;">
        <td> <img src='Figures/vasu.png'  width="300px" />   
    </tr>
</table>

Complétons le code suivant pour obtenir un oracle quantique qui soit l'analogue de $f(x,c)$

```Python
n=6
oracle=QCircuit(...)
for i in range(n):
    if c[i]==1:
        oracle.add(...)
```

## L'algorithme de Bernstein-Vazirani
L'algorithme de Bernstein-Vazirani fut publié en 1997 dans l'article "Quantum Complexity Theory". SIAM Journal on Computing. 

<table style="background: white;">
    <tr style="background: white;">
        <td> <img src='Figures/vazirani.png'  width="400px" />   
    </tr>
</table>

Implémentons l'algorithme et vérifions que l'algorithme retrouve la clé en un seul appel à l'oracle.

Indication: on pourra utiliser la commande 
```Python 
L=[H(i) for i in range(n+1)]
```
Pour générer $H^{\otimes n}$

L'état mesuré (State) correspond bien à la clé secrète $c$ (il faut négliger le premier bit de State qui est $0$ car nous utilisons un circuit à $n+1$ qubts mais le dernier qubit (donc le premier bit dans State) n'est jamais mesuré.

### Commentaires
L’algorithme de Bernstein–Vazirani résout en une seule itération un problème qui nécessiterait $n$requêtes classiques. Il y a ici un avantage en complexité en nombre de requêtes à l’oracle.

À retenir :

- superposition seule ne suffit pas
- l’interférence encode l’information utile
- la mesure lit un motif global
